# 🤖 Clinical Machine Learning Benchmarking, Evaluation & Explainable AI (XAI)

**Bachelor Capstone Project** • *Mohammad Hasan*

This notebook demonstrates the end-to-end modeling workflow using the `heart_risk` package:
1. Leak-Free Preprocessing Pipelines
2. Repeated Stratified Nested Cross-Validation (5-Fold × 3-Repeats)
3. Model Benchmarking & Statistical Evaluation
4. Probability Calibration & Brier Score
5. Decision Curve Analysis (DCA - Net Clinical Benefit)
6. Explainable AI (SHAP Waterfall Attributions)
7. Counterfactual "What-If" Lifestyle Simulations


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.heart_risk.config import load_config
from src.heart_risk.data.loader import load_dataset, get_data_splits
from src.heart_risk.features.pipeline import create_preprocessing_pipeline
from src.heart_risk.models.train import ModelTrainer
from src.heart_risk.evaluation.metrics import evaluate_classification_performance
from src.heart_risk.evaluation.clinical_utility import compute_decision_curve_analysis
from src.heart_risk.explainability.shap_engine import ClinicalExplainer
from src.heart_risk.explainability.counterfactual import CounterfactualSimulator
from src.heart_risk.ui.components import plot_waterfall_chart, plot_dca_curve

%matplotlib inline


## 1. Data Ingestion & Stratified Splits


In [ ]:
df = load_dataset()
X_train, X_test, y_train, y_test = get_data_splits(df, test_size=0.20, random_seed=42)
print(f"Train samples: {len(X_train)}, Test samples: {len(X_test)}")


## 2. Multi-Model Repeated Stratified Nested Cross-Validation Benchmark


In [ ]:
cfg = load_config()
trainer = ModelTrainer(cfg)

# Run full cross-validation benchmarking across all algorithms
benchmark_results = trainer.benchmark_all_models(df, save_results=True)

# Format summary table
summary = []
for name, res in benchmark_results.items():
    summary.append({
        'Algorithm': name.replace('_', ' ').title(),
        'ROC-AUC': f"{res['roc_auc_mean']:.3f} ± {res['roc_auc_std']:.3f}",
        'Sensitivity (Recall)': f"{res['sensitivity_mean']*100:.1f}%",
        'Specificity': f"{res['specificity_mean']*100:.1f}%",
        'Accuracy': f"{res['accuracy_mean']*100:.1f}%",
        'F1-Score': f"{res['f1_score_mean']:.3f}",
        'Brier Score': f"{res['brier_score_mean']:.3f}"
    })

pd.DataFrame(summary).sort_values(by='ROC-AUC', ascending=False)


## 3. Best Model Fitting & Holdout Test Evaluation


In [ ]:
best_pipeline, test_metrics = trainer.fit_and_calibrate_best_model(df)
print(f"🏆 Selected Best Model: {trainer.best_model_name_}")
print("\n📊 Holdout Test Performance Metrics:")
for k, v in test_metrics.items():
    if isinstance(v, float):
        print(f"  • {k}: {v:.4f}")
    elif k != 'confusion_matrix':
        print(f"  • {k}: {v}")


## 4. Decision Curve Analysis (DCA - Clinical Net Benefit)


In [ ]:
y_prob_test = best_pipeline.predict_proba(X_test)[:, 1]
dca = compute_decision_curve_analysis(y_test, y_prob_test)

fig_dca = plot_dca_curve(dca)
plt.show()


## 5. Explainable AI (XAI) - Patient Feature Attributions


In [ ]:
bg_df = X_train.copy()
explainer = ClinicalExplainer(best_pipeline, bg_df)

# Take a sample patient from test set
sample_patient = X_test.iloc[[0]]
explanation = explainer.explain_instance(sample_patient, top_k=7)

fig_waterfall = plot_waterfall_chart(
    explanation['top_contributions'],
    explanation['base_risk'],
    explanation['predicted_risk']
)
plt.show()


## 6. Counterfactual "What-If" Lifestyle Simulation


In [ ]:
simulator = CounterfactualSimulator(best_pipeline)
sim_res = simulator.simulate_interventions(sample_patient)

print(f"Baseline Risk: {sim_res['baseline_risk_pct']}%")
print(f"Optimized Target Risk: {sim_res['counterfactual_risk_pct']}%")
print(f"Absolute Risk Reduction: -{sim_res['absolute_risk_reduction_pct']}%")
print("\n🎯 Clinical Action Recommendations:")
for rec in sim_res['recommendations']:
    print(f"  • {rec.factor}: {rec.action_text} [Reduction: -{rec.risk_reduction_pct}%]")
